## Training of YoloV8 Model

### Imports

In [7]:
# %pip install ultralytics
from ultralytics import YOLO
from pathlib import Path

import torch 
import shutil
import itertools 
import pandas as pd


In [2]:
print("Torch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

model = YOLO("yolov8n.pt")
print("Model loaded")

Torch: 2.10.0+cu128
GPU available: True
Model loaded


### Parameter Tuning

In [4]:
# Below is the old model, changing some of the parameters in the function to get a better result!
# results = model.train(
#     data="yolov8_training_images/data.yaml",
#     epochs=50,
#     imgsz=640,
#     batch=8,
#     workers=0
# )

In [ ]:
# Comparing parameters and saving model to /weights/best path! 
# Decided to use yolov8n compared to yolov8s because 8n has higher performance for faster moving objects. Additionally, the robots
# are relatively large and easy to distinguish. 

def tune_yolov8(
        model_path="yolov8n.pt",
        data="yolov8_training_images/data.yaml",
        epoch_list=[20, 50, 100],
        lr_list=[0.01, 0.005, 0.001],
        batch_list=[8, 16],
        imgsz=640):

    results_summary = []

    best_map = 0
    best_model_path = None

    for epochs, lr, batch in itertools.product(epoch_list, lr_list, batch_list):

        print(f"\nTraining with epochs={epochs}, lr={lr}, batch={batch}")

        model = YOLO(model_path)

        results = model.train(
            data=data,
            epochs=epochs,
            imgsz=imgsz,
            batch=batch,
            workers=0,
            optimizer="SGD",
            lr0=lr,
            momentum=0.937,
            weight_decay=0.0005
        )

        metrics = model.val()

        map5095 = metrics.box.map

        results_summary.append({
            "epochs": epochs,
            "lr": lr,
            "batch": batch,
            "mAP50": metrics.box.map50,
            "mAP50-95": map5095
        })

        # Check if best model
        if map5095 > best_map:
            best_map = map5095

            best_weights = Path(results.save_dir) / "weights" / "best.pt"
            best_model_path = Path("best_tuned_yolov8.pt")

            shutil.copy(best_weights, best_model_path)

            print(f"New best model saved! mAP50-95 = {best_map:.4f}")

    df = pd.DataFrame(results_summary)

    print("\nBest configuration:")
    print(df.sort_values("mAP50-95", ascending=False).head())

    print(f"\nBest model saved at: {best_model_path}")

    return df

In [ ]:
# Once finished, this code chunk saves off the best model to best_tuned_yolov8.pt

results = tune_yolov8(
    epoch_list=[20, 50],
    lr_list=[0.01, 0.005, 0.001],
    batch_list=[8, 16]
)


Training with epochs=20, lr=0.01, batch=8
Ultralytics 8.4.21 🚀 Python-3.13.5 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100 80GB PCIe, 81152MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolov8_training_images/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train31, nbs=64, nms=False, opset=None, optimize=False, opti